# 現代的訓練迴圈：CIFAR10、混合精度、torch.compile

<a href="https://colab.research.google.com/github/tjwei/DataScienceSoftware2026/blob/main/05-pytorch/03-Modern-Training-Loop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

建議環境：Google Colab，`執行階段 → 變更執行階段類型 → T4 GPU`

In [ ]:
# 下載本章練習的參考解答 (q_*.py)
import os, urllib.request
BASE_URL = 'https://raw.githubusercontent.com/tjwei/DataScienceSoftware2026/main/05-pytorch/'
for fn in ['q_dataset.py', 'q_amp_benchmark.py', 'q_tta.py']:
    if not os.path.exists(fn):
        urllib.request.urlretrieve(BASE_URL + fn, fn)

In [ ]:
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from PIL import Image
from IPython.display import display
device = "cuda" if torch.cuda.is_available() else "cpu"
import os
NUM_WORKERS = 2 if os.name != "nt" else 0   # Colab (Linux) 用 2 個 worker；Windows 本機請設 0
torch.backends.cudnn.benchmark = True   # 輸入大小固定時，讓 cuDNN 自動挑最快的演算法

## CIFAR10

十個類別（飛機、汽車、鳥、貓、鹿、狗、青蛙、馬、船、卡車），32x32 彩色圖片。

In [ ]:
# 從 Hugging Face Hub 下載（比 CIFAR 原始網站快很多）
from datasets import load_dataset
from types import SimpleNamespace
def load_cifar10(split):
    ds = load_dataset("uoft-cs/cifar10", split=split)
    return SimpleNamespace(data=np.stack([np.array(im) for im in ds["img"]]), targets=list(ds["label"]))
train_raw = load_cifar10("train")
test_raw = load_cifar10("test")
name_array = np.array("飛機、汽車、鳥、貓、鹿、狗、青蛙、馬、船、卡車".split("、"))
train_raw.data.shape, len(train_raw.targets)

In [ ]:
def showX(X):
    """X: (N, 32, 32, 3) uint8"""
    display(Image.fromarray(np.concatenate(list(X), axis=1)).resize((64 * len(X), 64), Image.NEAREST))
showX(train_raw.data[:12])
print(name_array[train_raw.targets[:12]])

## Dataset 與 DataLoader

`Dataset` 只要實作 `__len__` 和 `__getitem__`；`DataLoader` 負責分 batch、打亂順序、多個 worker 平行讀取。

`torchvision.transforms.v2` 是新版的影像轉換 API。

In [ ]:
import torchvision.transforms.v2 as T
mean = np.array([0.4914, 0.4822, 0.4465])
std = np.array([0.2470, 0.2435, 0.2616])
train_tf = T.Compose([
    T.ToImage(),
    T.RandomCrop(32, padding=4, padding_mode="reflect"),
    T.RandomHorizontalFlip(),
    T.ToDtype(torch.float32, scale=True),
    T.Normalize(mean, std),
])
train_tf(train_raw.data[0]).shape

## Q
寫一個自己的 `Dataset` class，包裝 `train_raw.data`, `train_raw.targets` 與 `transform`，並且用 `DataLoader` 讀出一個 batch，檢查 shape。

In [ ]:
# %load q_dataset.py
%run -i q_dataset.py

### 把資料直接放在 GPU 上

Colab 只有 2 個 CPU 核心，用 CPU 做資料增強 (data augmentation) 常常會變成瓶頸，GPU 反而在等資料。
CIFAR10 很小 (50000x3x32x32)，可以整個放進 GPU，連資料增強也在 GPU 上做，速度會快很多。

In [ ]:
def to_tensor(raw):
    X = torch.tensor(raw.data).permute(0, 3, 1, 2).float().div(255)   # NHWC -> NCHW
    X = (X - torch.tensor(mean).view(1, 3, 1, 1).float()) / torch.tensor(std).view(1, 3, 1, 1).float()
    return X.to(device), torch.tensor(raw.targets).to(device)
train_X, train_y = to_tensor(train_raw)
test_X, test_y = to_tensor(test_raw)

def gpu_augment(X, pad=4):
    """random crop (reflect padding) + random horizontal flip, 全部在 GPU 上一次處理整個 batch"""
    N, C, H, W = X.shape
    Xp = F.pad(X, (pad,) * 4, mode="reflect")
    dx, dy = torch.randint(0, 2 * pad + 1, (2, N), device=X.device)
    rows = (torch.arange(H, device=X.device)[None, :] + dy[:, None])        # N x H
    cols = (torch.arange(W, device=X.device)[None, :] + dx[:, None])        # N x W
    X = Xp[torch.arange(N, device=X.device)[:, None, None, None],
           torch.arange(C, device=X.device)[None, :, None, None],
           rows[:, None, :, None], cols[:, None, None, :]]
    flip = torch.rand(N, device=X.device) < 0.5
    return torch.where(flip[:, None, None, None], X.flip(3), X)

def batches(X, y, batch_size, shuffle=False, augment=False):
    idx = torch.randperm(len(X), device=X.device) if shuffle else torch.arange(len(X), device=X.device)
    for i in range(0, len(X), batch_size):
        j = idx[i:i + batch_size]
        xb = X[j]
        yield (gpu_augment(xb) if augment else xb), y[j]

In [ ]:
xb, yb = next(batches(train_X, train_y, 12, augment=True))
img = (xb.permute(0, 2, 3, 1).cpu().numpy() * std + mean).clip(0, 1)
showX((img * 255).astype(np.uint8))

## 模型：小型 ResNet

* BatchNorm 讓訓練穩定
* Residual connection 讓深層網路也能訓練

In [ ]:
def conv_bn(c_in, c_out):
    return nn.Sequential(
        nn.Conv2d(c_in, c_out, 3, padding=1, bias=False),
        nn.BatchNorm2d(c_out),
        nn.GELU(),
    )

class Residual(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.block = nn.Sequential(conv_bn(c, c), conv_bn(c, c))
    def forward(self, x):
        return x + self.block(x)

def resnet9(num_classes=10, w=64):
    return nn.Sequential(
        conv_bn(3, w),
        conv_bn(w, 2 * w), nn.MaxPool2d(2), Residual(2 * w),      # 16x16
        conv_bn(2 * w, 4 * w), nn.MaxPool2d(2),                   # 8x8
        conv_bn(4 * w, 8 * w), nn.MaxPool2d(2), Residual(8 * w),  # 4x4
        nn.AdaptiveMaxPool2d(1), nn.Flatten(),
        nn.Linear(8 * w, num_classes),
    )
model = resnet9()
print(f"參數數量: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

## 混合精度訓練 (Automatic Mixed Precision, AMP)

* `torch.autocast`：在適合的運算 (matmul, conv) 自動改用 fp16，其他（例如 softmax、loss）保持 fp32
* `GradScaler`：fp16 能表示的範圍小，梯度太小會變成 0 (underflow)。先把 loss 放大再 backward，更新前再縮回來
* 如果是 A100/H100/RTX 30 以後的卡，可以改用 `dtype=torch.bfloat16`，就不需要 GradScaler。**T4 只能用 fp16**

In [ ]:
@torch.no_grad()
def evaluate(model, X=None, y=None, amp=True):
    X = test_X if X is None else X
    y = test_y if y is None else y
    model.eval()
    correct = 0
    for xb, yb in batches(X, y, 1000):
        with torch.autocast(device, dtype=torch.float16, enabled=amp):
            correct += (model(xb.contiguous(memory_format=torch.channels_last)).argmax(1) == yb).sum().item()
    return correct / len(X)

def fit(model, epochs=10, batch_size=512, lr=2e-3, amp=True, compile=False, label_smoothing=0.1):
    model = model.to(device, memory_format=torch.channels_last)   # channels_last 對 Tensor Core 卷積比較快
    train_model = torch.compile(model) if compile else model
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=5e-4, fused=(device == "cuda"))
    steps_per_epoch = (len(train_X) + batch_size - 1) // batch_size
    scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=lr, total_steps=epochs * steps_per_epoch)
    scaler = torch.amp.GradScaler(device, enabled=amp)
    for epoch in range(epochs):
        model.train()
        t0 = time.perf_counter()
        total_loss = 0
        for xb, yb in batches(train_X, train_y, batch_size, shuffle=True, augment=True):
            xb = xb.contiguous(memory_format=torch.channels_last)
            with torch.autocast(device, dtype=torch.float16, enabled=amp):
                loss = F.cross_entropy(train_model(xb), yb, label_smoothing=label_smoothing)
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            total_loss += loss.item()
        if device == "cuda": torch.cuda.synchronize()
        dt = time.perf_counter() - t0
        print(f"epoch {epoch:2d}  loss={total_loss/steps_per_epoch:.3f}  test acc={evaluate(model, amp=amp):.4f}"
              f"  {dt:.1f}s ({len(train_X)/dt:.0f} img/s)")
    return model

In [ ]:
model = fit(resnet9(), epochs=10)

10 個 epoch 應該可以到 90% 左右的準確率。

In [ ]:
# 看看預測結果
with torch.no_grad(), torch.autocast(device, dtype=torch.float16):
    pred = model(test_X[:12]).argmax(1).cpu()
showX(test_raw.data[:12])
print("預測:", name_array[pred])
print("答案:", name_array[test_raw.targets[:12]])

## Q
比較下面幾種設定，每種只跑 2 個 epoch，看看每秒可以處理幾張圖 (img/s)：
* `amp=False`（純 fp32）
* `amp=True`
* `amp=True, compile=True`（第一個 epoch 包含編譯時間，看第二個 epoch）

`torch.compile` 會把模型編譯成最佳化過的 kernel (使用 Triton)，第一次執行比較慢。

注意：最佳化**一定要實際量測**。在 T4 上，AMP 大約快 3~4 倍；但這個小模型用 `torch.compile` 反而可能變慢
（cuDNN 的卷積已經很快，T4 的 SM 數量也不足以使用某些自動調校模式）。大模型、或有很多小運算 (elementwise ops) 的模型，compile 的效果通常比較明顯。

In [ ]:
# %load q_amp_benchmark.py
%run -i q_amp_benchmark.py

## 存檔 checkpoint

除了模型參數，若要能「接著訓練」，也要存 optimizer、scheduler、scaler 的狀態。
這裡示範最基本的存模型參數。

In [ ]:
torch.save(model.state_dict(), "resnet9_cifar10.pt")
model2 = resnet9().to(device, memory_format=torch.channels_last)
model2.load_state_dict(torch.load("resnet9_cifar10.pt"))
evaluate(model2)

## Q：Test Time Augmentation (TTA)
預測時，把原圖和水平翻轉的圖都丟進模型，把兩個輸出 (logits 或機率) 平均。準確率有提升嗎？

In [ ]:
# %load q_tta.py
%run -i q_tta.py